In [0]:
%pip install azure-eventhub

In [0]:
dbutils.library.restartPython()

In [0]:
#!pip install pandas

In [0]:
# Cell 3 — now your actual producer code starts
import pandas as pd
import time
import os
from azure.eventhub import EventHubProducerClient, EventData

conn_str = "Endpoint=sb://retailrocket-events.servicebus.windows.net/;SharedAccessKeyName=producer-send-policy;SharedAccessKey=9ODdv6qRT4cceSACx9AyQ8ExeCu+9VHRI+AEhNPWhy8="
eventhub_name = "retailrocket-events"

checkpoint_dir = "/Volumes/retrailrocket/landing/raw/Streaming/Checkpoint"
os.makedirs(checkpoint_dir, exist_ok=True)
# Load the events file from your Volume — sort by timestamp so replay order matches real event order
progress_file = "/Volumes/retrailrocket/landing/raw/Streaming/Checkpoint/producer_progress.txt"
events_path = "/Volumes/retrailrocket/landing/raw/Streaming/events.csv"

batch_size = 100
sleep_seconds = 0.5

# --- Load and sort events ---
events_df = pd.read_csv(events_path)
events_df = events_df.sort_values("timestamp").reset_index(drop=True)
total_rows = len(events_df)

# --- Determine where to resume from ---
if os.path.exists(progress_file):
    with open(progress_file, "r") as f:
        start_row = int(f.read().strip())
    print(f"Resuming from row {start_row} of {total_rows}")
else:
    start_row = 0
    print(f"Starting fresh from row 0 of {total_rows}")

if start_row >= total_rows:
    print("All rows already sent. Nothing to do. Delete the progress file to replay from the beginning.")
else:
    producer = EventHubProducerClient.from_connection_string(
        conn_str=conn_str,
        eventhub_name=eventhub_name
    )

    with producer:
        for start in range(start_row, total_rows, batch_size):
            batch = producer.create_batch()
            chunk = events_df.iloc[start:start + batch_size]

            for _, row in chunk.iterrows():
                event_json = row.to_json()
                try:
                    batch.add(EventData(event_json))
                except ValueError:
                    producer.send_batch(batch)
                    batch = producer.create_batch()
                    batch.add(EventData(event_json))

            producer.send_batch(batch)
            rows_sent_so_far = start + len(chunk)
            print(f"Sent events {start} to {rows_sent_so_far}")

            # Save progress only AFTER a successful send
            with open(progress_file, "w") as f:
                f.write(str(rows_sent_so_far))

            time.sleep(sleep_seconds)

    print("Replay complete.")